# 08 — Locked Official Test Evaluation

This is the first notebook allowed to read the official test split. It performs **no fitting, feature selection, GridSearchCV, or threshold tuning**. It loads the frozen Logistic Regression model and frozen schema, extracts the already-defined features from the official test images, and reports locked test metrics.

In [1]:
from pathlib import Path
import sys,json,joblib,gc
import pandas as pd
from tqdm import tqdm
ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/"data").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATASET=ROOT/"data/raw/deepfake_merged_dataset"; assert DATASET.exists(),f"Missing dataset: {DATASET}"
from src.features import extract_all
from src.modeling import evaluate
s=json.loads((ROOT/"models/feature_schema_l1.json").read_text()); model=joblib.load(ROOT/"models/model_frozen.joblib",mmap_mode="r")
selected=s["selected_features"]; rows=[]; labels=[]
IMAGE_EXTENSIONS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
for cls,y in [("real",0),("fake",1)]:
    paths=sorted(p for p in (DATASET/"test"/cls).iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    print(cls,"images:",len(paths))
    for p in tqdm(paths,desc=cls): rows.append(extract_all(p)); labels.append(y)
X=pd.DataFrame(rows)
X=X.reindex(columns=selected)
p=model.predict_proba(X)[:,1]
metrics=evaluate(labels,p,float(s["threshold"]))
print("LOCKED OFFICIAL TEST METRICS:"); print(json.dumps(metrics,indent=2))
out=ROOT/"metrics/locked_test_metrics.json"; out.write_text(json.dumps(metrics,indent=2),encoding="utf-8"); print("Saved:",out)


C:\Users\devar\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:137: UserWarning: mmap_mode "r" is not compatible with compressed file d:\deepfake_noise_wavelet_ml\models\model_frozen.joblib. "r" flag will be ignored.
  return next(self.gen)


real images: 11199


real: 100%|██████████| 11199/11199 [09:47<00:00, 19.08it/s]


fake images: 14305


fake: 100%|██████████| 14305/14305 [11:50<00:00, 20.13it/s]


LOCKED OFFICIAL TEST METRICS:
{
  "roc_auc": 0.6622966567238879,
  "pr_auc": 0.6674567319037547,
  "accuracy": 0.6231571518193224,
  "precision": 0.6152637265494548,
  "recall": 0.8757777001048584,
  "f1": 0.7227622811319121,
  "balanced_accuracy": 0.5881254783228105,
  "tn": 3365,
  "fp": 7834,
  "fn": 1777,
  "tp": 12528
}
Saved: d:\deepfake_noise_wavelet_ml\metrics\locked_test_metrics.json


d:\deepfake_noise_wavelet_ml\.venv\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
